# 📚 [4주차] 데이터 분석 전처리반(판다스) 복습과제
### 작성자: 이창현
---
### 📌 복습 키워드
- MCAR / MAR / MNAR
- 결측치 처리: 보간 / 삭제 / IPW
- 독립변수 vs 종속변수 보간
- 자료형 보간

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (Mac/Windows 공용)
plt.rcParams['font.family'] = 'AppleGothic'  # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료 ✅')

---
## 1️⃣ 결측치 유형 : MCAR / MAR / MNAR

| 유형 | 풀네임 | 의미 | 예시 |
|------|--------|------|------|
| **MCAR** | Missing Completely At Random | 결측이 완전히 무작위 | 설문지 랜덤 누락 |
| **MAR** | Missing At Random | 다른 변수와 관련된 무작위 결측 | 나이가 많을수록 소득 미기재 |
| **MNAR** | Missing Not At Random | 결측 자체가 값과 관련 | 소득 낮을수록 소득 미기재 |

In [ ]:
# 예제 데이터 생성
np.random.seed(42)

df = pd.DataFrame({
    'age'   : [25, 32, 45, 28, 55, 38, 60, 22, 47, 35],
    'income': [3000, 4500, 6000, 3200, 7000, 5000, 8000, 2800, 6500, 4800],
    'score' : [80, 75, 90, 85, 70, 88, 65, 92, 78, 83]
})

print('원본 데이터')
print(df)
print(f'\n데이터 shape: {df.shape}')

In [ ]:
# ① MCAR: 완전 무작위 결측 (랜덤하게 NaN 삽입)
df_mcar = df.copy()
random_idx = np.random.choice(df.index, size=3, replace=False)
df_mcar.loc[random_idx, 'score'] = np.nan

print('MCAR 예시 - score 컬럼 무작위 결측')
print(df_mcar)
print(f'\n결측치 개수:\n{df_mcar.isnull().sum()}')

In [ ]:
# ② MAR: 다른 변수(age)와 관련된 결측
# age >= 45인 경우 income 결측 (나이 많을수록 소득 미기재)
df_mar = df.copy()
df_mar.loc[df_mar['age'] >= 45, 'income'] = np.nan

print('MAR 예시 - age >= 45 이면 income 결측')
print(df_mar)
print(f'\n결측치 개수:\n{df_mar.isnull().sum()}')

In [ ]:
# ③ MNAR: 결측값 자체가 값과 관련
# income이 낮을수록(3500 미만) income 미기재
df_mnar = df.copy()
df_mnar.loc[df_mnar['income'] < 3500, 'income'] = np.nan

print('MNAR 예시 - income < 3500 이면 income 결측')
print(df_mnar)
print(f'\n결측치 개수:\n{df_mnar.isnull().sum()}')

---
## 2️⃣ 결측치 처리 방법 : 삭제

| 방법 | 함수 | 설명 |
|------|------|------|
| 행 삭제 | `dropna(axis=0)` | 결측치 있는 행 전체 삭제 |
| 열 삭제 | `dropna(axis=1)` | 결측치 있는 열 전체 삭제 |
| thresh | `dropna(thresh=n)` | n개 이상 유효값 있는 행만 유지 |

> ⚠️ **주의**: 데이터 손실 위험 → MCAR일 때만 권장

In [ ]:
# 결측치 있는 데이터 준비
df_del = df_mcar.copy()
print(f'삭제 전 shape: {df_del.shape}')
print(df_del)

# 행 삭제
df_dropped = df_del.dropna(axis=0)
print(f'\n행 삭제 후 shape: {df_dropped.shape}')
print(df_dropped)

---
## 3️⃣ 결측치 처리 방법 : 보간

| 방법 | 설명 | 적합한 경우 |
|------|------|-------------|
| `fillna(mean)` | 평균값으로 대체 | 수치형, MCAR |
| `fillna(median)` | 중앙값으로 대체 | 이상치 있을 때 |
| `fillna(mode)` | 최빈값으로 대체 | 범주형 |
| `interpolate()` | 선형 보간 | 시계열 데이터 |
| `ffill()` | 앞 값으로 채우기 | 시계열 |
| `bfill()` | 뒤 값으로 채우기 | 시계열 |

In [ ]:
df_fill = df_mcar.copy()

# ① 평균값 보간
mean_val = df_fill['score'].mean()
df_mean = df_fill.copy()
df_mean['score'] = df_mean['score'].fillna(mean_val)
print(f'평균값 보간 (mean={mean_val:.2f})')
print(df_mean['score'].values)

# ② 중앙값 보간
median_val = df_fill['score'].median()
df_median = df_fill.copy()
df_median['score'] = df_median['score'].fillna(median_val)
print(f'\n중앙값 보간 (median={median_val:.2f})')
print(df_median['score'].values)

# ③ 선형 보간
df_interp = df_fill.copy()
df_interp['score'] = df_interp['score'].interpolate(method='linear')
print(f'\n선형 보간')
print(df_interp['score'].values)

In [ ]:
# ④ ffill / bfill 비교
df_ff = df_fill.copy()
df_bf = df_fill.copy()

df_ff['score'] = df_ff['score'].ffill()
df_bf['score'] = df_bf['score'].bfill()

print('ffill (앞 값으로 채우기)')
print(df_ff['score'].values)
print('\nbfill (뒤 값으로 채우기)')
print(df_bf['score'].values)

---
## 4️⃣ 결측치 처리 방법 : IPW (Inverse Probability Weighting)

- **개념**: 결측이 발생할 확률의 역수를 가중치로 부여
- **목적**: MAR 상황에서 편향(bias) 없이 모집단 추정
- **핵심 아이디어**:
  - 결측 확률이 높은 그룹 → 가중치 높게
  - 결측 확률이 낮은 그룹 → 가중치 낮게

> 💡 **수식**: `IPW = 1 / P(관측됨)`

In [ ]:
from sklearn.linear_model import LogisticRegression

# MAR 데이터 사용
df_ipw = df_mar.copy()

# ① 결측 여부 컬럼 생성 (1=관측됨, 0=결측)
df_ipw['observed'] = df_ipw['income'].notna().astype(int)

# ② 로지스틱 회귀로 관측될 확률 추정
X = df_ipw[['age']]
y = df_ipw['observed']

lr = LogisticRegression()
lr.fit(X, y)

# ③ 관측 확률 계산
df_ipw['prob_observed'] = lr.predict_proba(X)[:, 1]

# ④ IPW 계산 (관측된 행만)
df_ipw['ipw'] = np.where(
    df_ipw['observed'] == 1,
    1 / df_ipw['prob_observed'],
    np.nan
)

print('IPW 계산 결과')
print(df_ipw[['age', 'income', 'observed', 'prob_observed', 'ipw']])

# ⑤ IPW 가중 평균으로 income 추정
observed_df = df_ipw[df_ipw['observed'] == 1]
weighted_mean = np.average(observed_df['income'], weights=observed_df['ipw'])
simple_mean   = observed_df['income'].mean()

print(f'\n단순 평균: {simple_mean:.2f}')
print(f'IPW 가중 평균: {weighted_mean:.2f}')
print(f'실제 전체 평균: {df["income"].mean():.2f}')

---
## 5️⃣ 독립변수 vs 종속변수 보간

| 구분 | 설명 | 결측 처리 방법 |
|------|------|----------------|
| **독립변수 (X)** | 예측에 사용하는 변수 | 평균/중앙값/모델 기반 보간 |
| **종속변수 (y)** | 예측 대상 변수 | 원칙적으로 삭제 권장 |

> ⚠️ **종속변수(y) 결측을 보간하면 데이터 누수(Data Leakage) 위험!**

In [ ]:
# 독립변수(X) 결측 → 보간 권장
df_x = df.copy()
df_x.loc[[2, 5], 'age'] = np.nan  # 독립변수 결측

print('독립변수(age) 결측 → 중앙값 보간')
df_x['age'] = df_x['age'].fillna(df_x['age'].median())
print(df_x[['age', 'income', 'score']])

print('\n' + '='*50)

# 종속변수(y) 결측 → 삭제 권장
df_y = df.copy()
df_y.loc[[3, 7], 'score'] = np.nan  # 종속변수 결측

print('\n종속변수(score) 결측 → 행 삭제 권장')
df_y_clean = df_y.dropna(subset=['score'])
print(f'삭제 전: {df_y.shape[0]}행 → 삭제 후: {df_y_clean.shape[0]}행')
print(df_y_clean)

---
## 6️⃣ 자료형 보간

| 자료형 | 권장 보간 방법 | 이유 |
|--------|----------------|------|
| **수치형 (int/float)** | 평균, 중앙값, 선형보간 | 연속적 값 |
| **범주형 (object)** | 최빈값(mode), 'Unknown' | 평균 계산 불가 |
| **날짜형 (datetime)** | ffill, bfill, 선형보간 | 시간 순서 의미 있음 |
| **불리언 (bool)** | 최빈값, False | 이진 값 |

In [ ]:
# 자료형별 보간 예제
df_types = pd.DataFrame({
    'name'    : ['Alice', 'Bob', None, 'Dave', 'Eve'],        # 범주형
    'age'     : [25, None, 35, 28, None],                     # 수치형
    'join_date': pd.to_datetime(['2023-01-01', '2023-02-01',
                                  None, '2023-04-01', None]), # 날짜형
    'is_vip'  : [True, None, False, True, None]               # 불리언
})

print('원본 데이터')
print(df_types)
print(f'\n자료형:\n{df_types.dtypes}')

In [ ]:
df_filled = df_types.copy()

# ① 수치형 → 평균 보간
df_filled['age'] = df_filled['age'].fillna(df_filled['age'].mean())

# ② 범주형 → Unknown으로 대체
df_filled['name'] = df_filled['name'].fillna('Unknown')

# ③ 날짜형 → ffill
df_filled['join_date'] = df_filled['join_date'].ffill()

# ④ 불리언 → 최빈값
mode_vip = df_filled['is_vip'].mode()[0]
df_filled['is_vip'] = df_filled['is_vip'].fillna(mode_vip)

print('자료형별 보간 완료')
print(df_filled)
print(f'\n결측치 확인:\n{df_filled.isnull().sum()}')

---
## 📊 전체 요약 정리

| 키워드 | 핵심 내용 |
|--------|----------|
| **MCAR** | 완전 무작위 결측 → 삭제/보간 모두 가능 |
| **MAR** | 다른 변수와 관련된 결측 → IPW, 모델 기반 보간 |
| **MNAR** | 결측값 자체와 관련 → 가장 처리 어려움, 도메인 지식 필요 |
| **보간** | 평균/중앙값/최빈값/선형보간/ffill/bfill |
| **삭제** | dropna() → MCAR일 때만 권장 |
| **IPW** | MAR에서 편향 없는 추정 → 역확률 가중치 |
| **독립변수** | 보간 권장 (정보 보존) |
| **종속변수** | 삭제 권장 (Data Leakage 방지) |
| **자료형** | 수치형→평균, 범주형→최빈값, 날짜→ffill |